# Introduction

This workbook documents the data cleaning and preprocessing steps applied to the Amazon Reviews dataset prior to sentiment analysis and suspicious review detection. The notebook performs structured data inspection, handles missing values, standardizes key variables, removes unusable observations, and engineers additional features required for machine learning tasks. 

The goal of this workflow is to produce a clean, reliable, and analysis ready dataset while maintaining transparency and reproducibility of the preprocessing pipeline.

In [32]:
#Install Necessary Libraries
!pip install spacy

!python -m spacy download en_core_web_sm
!pip install nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 48.3 MB/s  0:00:00eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [33]:
# Import Neccessary Libraries
import pandas as pd
import numpy as np
import spacy
import re
import nltk

In [34]:
# Load and preview the data
data = pd.read_csv("amazon_reviews_categories.csv")
print("Shape:", data.shape)

data.head()

Shape: (829621, 11)


,asin,category,reviewerID,reviewerName,overall,summary,reviewText,unixReviewTime,reviewTime,verified,vote
0,B00CYQP3AK,Amazon Devices & Accessories,A3UXVNOFC8TJPT,Bkonasek,5.0,Great product,I received this as a birthday present. I am do...,1382140800,"10 19, 2013",False,2
1,B00CYQP3AK,Amazon Devices & Accessories,AC2VR9U3NN2UD,tikitoo,2.0,Battery Issues and Light bleed - 2nd time Not ...,Battery will not charge to 100%. When I receiv...,1382140800,"10 19, 2013",True,22
2,B00CYQP3AK,Amazon Devices & Accessories,A14D0LJ1YRLLCI,gabbie03,1.0,Missing promised features. Not ready for prime...,Buyer Beware. these units are being shipped ri...,1382140800,"10 19, 2013",True,54
3,B00CYQP3AK,Amazon Devices & Accessories,A3FVF1JIOA5GEK,MN_Ranger,5.0,Amazing Performance Worthy of Upgrade,"Netflix Issue:\nWhen I first got the KFHDX, I ...",1382054400,"10 18, 2013",True,577
4,B00CYQP3AK,Amazon Devices & Accessories,A3KP101J8H7EWY,L.D.K,5.0,This is by far the best tablet for the price,"I have owned many versions of the Kindle, and ...",1382054400,"10 18, 2013",True,8


The dataset has 829,621 rows and 11 columns.

## Data Cleaning Phase one

## Inspect Dataset Columns
Check the column names in the dataset.  
This helps confirm that all required variables are present before applying the cleaning steps.


In [35]:
print(data.columns.tolist())

['asin', 'category', 'reviewerID', 'reviewerName', 'overall', 'summary', 'reviewText', 'unixReviewTime', 'reviewTime', 'verified', 'vote']


## Handle duplicates

During the exploratory data analysis, duplicate records were identified in the dataset. These duplicates likely resulted from the data collection process, where the same review may have been captured multiple times during web scraping.

To address this issue, duplicates were removed nd retained only the first occurrence of reviews with identical asin, reviewerID, reviewText, and unixReviewTime.

In [36]:

# Remove duplicates based on specific columns
reviews_clean = data.drop_duplicates(
    subset=['asin', 'reviewerID', 'reviewText', 'unixReviewTime'],
    keep='first'
)

#  reset index
reviews_clean = reviews_clean.reset_index(drop=True)

## Vlidate duplicate removal

In [37]:
distinct_text = (
    reviews_clean
    .groupby(['asin', 'reviewerID', 'category'])
    .agg(
        distinct_texts=('reviewText', 'nunique'),
        total_rows=('reviewText', 'size')
    )
    .reset_index()
)

distinct_text = distinct_text[distinct_text['total_rows'] > 1]

# Order and limit
distinct_text = distinct_text.sort_values('total_rows', ascending=False).head(50)

distinct_text

,asin,reviewerID,category,distinct_texts,total_rows
93090,0099911701,A1L43KWWR05PCS,Books,6,7
498669,B000W5QSYA,A1AG2S30FR8M8B,Pet Supplies,6,6
503200,B000W5QSYA,A2OY938ZNE1GTB,Pet Supplies,6,6
106953,0099911701,A3CMVOJCGJRZSP,Books,4,5
499099,B000W5QSYA,A1FCIAE7P5EB7Q,Pet Supplies,5,5
506822,B000W5QSYA,A3TAS1AG6FMBQW,Pet Supplies,5,5
502101,B000W5QSYA,A2C99XQBQO17H8,Pet Supplies,5,5
507744,B000W5QSYA,ABAHKMFEHT8K1,Pet Supplies,5,5
91164,0099911701,A1CIMFTHR43Z39,Books,5,5
501389,B000W5QSYA,A24DALFMRFB00J,Pet Supplies,4,4


## Investigate remaining anomalies
Some reviews still shared identical timestamps but had different text.

In [38]:
review_sample1 = (
    reviews_clean[
        (reviews_clean['asin'] == '0099911701') &
        (reviews_clean['reviewerID'] == 'A1L43KWWR05PCS')
    ]
    .sort_values('reviewTime')
    .head(10)
)

review_sample1

,asin,category,reviewerID,reviewerName,overall,summary,reviewText,unixReviewTime,reviewTime,verified,vote
379153,0099911701,Books,A1L43KWWR05PCS,lawyeraau,5.0,RIVETING TIME TRAVEL ADVENTURE...,"A lover of historical fiction, adventure books...",1206921600,"03 31, 2008",False,8
379151,0099911701,Books,A1L43KWWR05PCS,lawyeraau,5.0,A LOVE THAT TRANSCENDS TIME CONTINUES TO RIVET...,"This is the second in a series of time travel,...",1207094400,"04 2, 2008",True,NaN
379450,0099911701,Books,A1L43KWWR05PCS,lawyeraau,5.0,A LOVE THAT TRANSCENDS TIME CONTINUES TO RIVET...,"This is the second in a series of time travel,...",1145750400,"04 23, 2006",True,NaN
379451,0099911701,Books,A1L43KWWR05PCS,lawyeraau,5.0,AN ABSOLUTELY RIVETING STORY!!!!!...,"A lover of historical fiction, adventure books...",1145750400,"04 23, 2006",False,4
380877,0099911701,Books,A1L43KWWR05PCS,lawyeraau,5.0,AN ABSOLUTELY RIVETING STORY!!!!!,"A lover of historical fiction, adventure books...",988588800,"04 30, 2001",True,40
380876,0099911701,Books,A1L43KWWR05PCS,lawyeraau,5.0,A LOVE THAT TRANSCENDS TIME CONTINUES TO RIVET...,"This is the second in a series of time travel,...",989020800,"05 5, 2001",True,225
380513,0099911701,Books,A1L43KWWR05PCS,lawyeraau,5.0,AN ABSOLUTELY RIVETING STORY!!!!!!,"A lover of historical fiction, adventure books...",1028332800,"08 3, 2002",True,NaN




After removing exact duplicates, further inspection revealed another pattern where multiple reviews shared the same timestamp but had different review texts and metadata.

These were not true duplicates but likely represent different snapshots of the same review captured during the web crawling process. 

The identical timestamp reviews observed during analysis were not classified as burst behaviour. Therefore, they were treated as data collection artifacts rather than genuine reviewer behaviour.


Since these rows contain different textual information, they were retained in the dataset and treated as potential data collection artifacts rather than genuine duplicate records.

## Missing Values Summary
Before cleaning the data, we identify which columns contain missing values.
This step helps determine which preprocessing strategies are needed.



In [39]:
data.isnull().sum()

asin                   0
category               0
reviewerID             0
reviewerName          93
overall                0
summary              365
reviewText           502
unixReviewTime         0
reviewTime             0
verified               0
vote              792455
dtype: int64

In [40]:
# Get all text columns in the dataset
text_columns = data.select_dtypes(include="object").columns

for col in text_columns:
    
    # Count NULL values
    null_count = data[col].isna().sum()
    
    # Count empty or whitespace values
    empty_or_whitespace_count = data[col].fillna("").astype(str).str.strip().eq("").sum()
    print(f"Column being checked: {col}")
    print(f"Number of NULL values: {null_count}")
    print(f"Number of empty/whitespace values: {empty_or_whitespace_count}")
    print()

Column being checked: asin
Number of NULL values: 0
Number of empty/whitespace values: 0

Column being checked: category
Number of NULL values: 0
Number of empty/whitespace values: 0

Column being checked: reviewerID
Number of NULL values: 0
Number of empty/whitespace values: 0

Column being checked: reviewerName
Number of NULL values: 93
Number of empty/whitespace values: 100

Column being checked: summary
Number of NULL values: 365
Number of empty/whitespace values: 372

Column being checked: reviewText
Number of NULL values: 502
Number of empty/whitespace values: 505

Column being checked: reviewTime
Number of NULL values: 0
Number of empty/whitespace values: 0

Column being checked: vote
Number of NULL values: 792455
Number of empty/whitespace values: 792455



A dataset wide check was performed to identify NULL values and whitespace only entries across all text columns. The columns asin, category, reviewerID, and reviewTime were found to have no missing or empty values. However, small issues were observed in reviewerName, summary, and reviewText, where some entries were either missing or contained only whitespace. Since reviewerName and summary are not critical for analysis, their missing values can be replaced with placeholders. Rows with missing or empty reviewText will be removed because the review text is essential for sentiment analysis.

## Handle Missing Reviewer Names

Some reviews have missing values in the reviewerName column.
Since reviewerID uniquely identifies each reviewer, missing names do not affect analysis.  
Therefore, missing reviewer names are replaced with the placeholder value Unknown.

In [41]:
## replace empty or whitespace reviewer names with "Unknown"
data["reviewerName"] = data["reviewerName"].replace(r'^\s*$', "Unknown", regex=True)

# fill missing review names with Unknown
data["reviewerName"] = data["reviewerName"].fillna("Unknown")

#check if there are still missing values
data["reviewerName"].isnull().sum()

np.int64(0)

The result returned 0, indicating that there are no missing values and whitespaces remaining in the reviewerName column after replacing null values with "Unknown"

## Handle the summary column

In [42]:
# drop summary column

data = data.drop(columns=["summary"])

The summary column was removed because it contains only short titles or brief descriptions of reviews. These summaries often duplicate the information already present in the reviewText column and do not provide significant additional context for analysis. Since the full review text contains richer information that is more useful for sentiment analysis and modeling, the summary column was dropped to simplify the dataset.

## Handle the review text column



### Preview examples of problematic reviews

This step displays rows where the review text is missing or empty so that the cleaning logic can be verified before removal.

In [43]:
# Review text column name
review_col = "reviewText"

# Show rows with empty or whitespace only reviewText
print("Examples of empty or whitespace only reviewText rows:")

display(data[data[review_col].fillna("").astype(str).str.strip() == ""].head())

Examples of empty or whitespace only reviewText rows:


,asin,category,reviewerID,reviewerName,overall,reviewText,unixReviewTime,reviewTime,verified,vote
1247,B00CYQP3AK,Amazon Devices & Accessories,A255DINSOQKQSD,Arlene,4.0,NaN,1444435200,"10 10, 2015",True,NaN
2467,B00CYQP3AK,Amazon Devices & Accessories,A2K3RKKX4CTEOV,Dan Hoveland,4.0,NaN,1438646400,"08 4, 2015",True,3
5325,B00CYQP3AK,Amazon Devices & Accessories,A2SLRLBSYKWK12,Liz rojas,4.0,NaN,1429833600,"04 24, 2015",True,NaN
8892,B00CYQP3AK,Amazon Devices & Accessories,A2DIKHN8M45SK,Haley,5.0,NaN,1422835200,"02 2, 2015",True,NaN
10100,B00CYQP3AK,Amazon Devices & Accessories,A3M8XWSNL9KB8A,Dyamond,5.0,NaN,1421452800,"01 17, 2015",True,NaN


## Remove Reviews with Missing Text

The reviewText column contains the main textual content of each review.  
Since sentiment analysis relies on review text, rows with missing reviewText cannot be used.
Therefore, all rows with missing review text are removed from the dataset.

In [44]:
# remove rows where reviewText is missing (NaN)
data = data[data["reviewText"].notna()]

# remove rows where reviewText is empty or contains only whitespace
data = data[data["reviewText"].astype(str).str.strip() != ""]

print("Shape after removing missing reviewText:", data.shape)

Shape after removing missing reviewText: (829116, 10)


The summary column contains short review titles that often duplicate the sentiment already expressed in the rating or review text. Since the reviewText column provides richer textual information for analysis, the summary column will not be used in the modeling process

## Handle the Vote columns

The vote column represents the number of helpfulness votes a review received.
Inspect a sample of this column to understand its format.  
Some values may contain commas (e.g., "1,234"), which must be cleaned before converting the column to numeric format.

In [45]:
data["vote"].head(10)

0        2
1       22
2       54
3      577
4        8
5       94
6       16
7       25
8      376
9    4,317
Name: vote, dtype: object

A sample of the vote column was inspected to understand the structure of the helpfulness vote values. The column contains numeric values representing how many users found a review helpful. However, some values contain commas (e.g., "4,317"), indicating that the column is currently stored as text (object type) rather than a numeric format.

Because commas prevent direct numeric conversion, they must be removed before converting the column to a numeric data type for analysis.

###   Clean the Vote column 

In [46]:
# Inspect vote column
data["vote"].sample(10)

# Convert to string and remove whitespace
data["vote"] = data["vote"].astype(str).str.strip()

# Convert vote column to numeric
data["vote"] = pd.to_numeric(data["vote"], errors="coerce")

# Fill missing values with zero
data["vote"] = data["vote"].fillna(0)

# Convert to integer
data["vote"] = data["vote"].astype(int)

# Verify conversion
data["vote"].dtype

dtype('int64')

## Validate the cleaning worked

In [47]:
text_columns = data.select_dtypes(include="object").columns

for col in text_columns:
    null_count = data[col].isna().sum()
    empty_or_whitespace_count = data[col].fillna("").astype(str).str.strip().eq("").sum()
    
    print(f"Column being checked: {col}")
    print(f"Number of NULL values: {null_count}")
    print(f"Number of empty/whitespace values: {empty_or_whitespace_count}")
    print()

Column being checked: asin
Number of NULL values: 0
Number of empty/whitespace values: 0

Column being checked: category
Number of NULL values: 0
Number of empty/whitespace values: 0

Column being checked: reviewerID
Number of NULL values: 0
Number of empty/whitespace values: 0

Column being checked: reviewerName
Number of NULL values: 0
Number of empty/whitespace values: 0

Column being checked: reviewText
Number of NULL values: 0
Number of empty/whitespace values: 0

Column being checked: reviewTime
Number of NULL values: 0
Number of empty/whitespace values: 0



## Create Feature: has_vote
A new binary feature has_vote is created to indicate whether a review received any helpfulness votes.

The feature is defined as 0 the review received no votes, 1 the review received at least one vote
This feature can be useful for suspicious review detection and modeling.

In [48]:
data["has_vote"] = (data["vote"] > 0).astype(int)

data[["vote", "has_vote"]].head()

,vote,has_vote
0,2,1
1,22,1
2,54,1
3,577,1
4,8,1


## Create review length features

Three features are created from the cleaned reviewText column:

- review_length: number of characters in the review
- word_count: number of words in the review
- short_review_flag: indicates whether the review has 10 characters or fewer

These features are useful for identifying very short reviews and for supporting later text analysis tasks.

In [49]:
# Convert reviewText to string
data[review_col] = data[review_col].astype(str)

# Create review_length
data["review_length"] = data[review_col].str.len()

# Create word_count
data["word_count"] = data[review_col].str.split().str.len()

# Create short_review_flag
data["short_review_flag"] = np.where(data["review_length"] <= 10, 1, 0)

print("New features created successfully.")
data[[review_col, "review_length", "word_count", "short_review_flag"]].head(10)

New features created successfully.


,reviewText,review_length,word_count,short_review_flag
0,I received this as a birthday present. I am do...,130,22,0
1,Battery will not charge to 100%. When I receiv...,1693,325,0
2,Buyer Beware. these units are being shipped ri...,1051,198,0
3,"Netflix Issue:\nWhen I first got the KFHDX, I ...",31090,5690,0
4,"I have owned many versions of the Kindle, and ...",311,58,0
5,"SUMMARY - Overall, I really like the Kindle Fi...",13276,2510,0
6,"We received the Kindle Fire HDX 7"" about a wee...",859,152,0
7,"I have a first generation Kindle Fire, it did ...",621,124,0
8,This device is a breathtaking movie watching d...,5059,914,0
9,To sum up what I will tell you about in the de...,10538,1991,0


In [50]:
##Show examples of short reviews
# Filter short reviews
short_reviews_df = data[data["short_review_flag"] == 1]

print("Number of short reviews (<=10 characters):", len(short_reviews_df))
display(short_reviews_df[[review_col, "review_length", "word_count", "short_review_flag"]].head(10))

Number of short reviews (<=10 characters): 39131


,reviewText,review_length,word_count,short_review_flag
22,Great,5,1,1
44,Works good,10,2,1
83,It's nice,9,2,1
115,AWESOME!,8,1,1
119,good,4,1,1
146,LOVE IT,7,2,1
152,Good,4,1,1
214,On it now,9,3,1
243,Love it,7,2,1
265,Love it,7,2,1


## Validation
This step confirms that the dataset is clean and ready for downstream tasks such as sentiment analysis and suspicious review detection.

In [51]:
data.isnull().sum()

asin                 0
category             0
reviewerID           0
reviewerName         0
overall              0
reviewText           0
unixReviewTime       0
reviewTime           0
verified             0
vote                 0
has_vote             0
review_length        0
word_count           0
short_review_flag    0
dtype: int64

## Data cleaning Phase two

The goal of the data cleaning phase is to prepare raw review text for analysis by removing noise and standardizing the text format. Raw textual data often contains HTML tags, punctuation, irregular spacing, and other unwanted characters that can negatively affect Natural Language Processing (NLP) models.

Cleaning the data ensures that the text is consistent, readable, and suitable for feature extraction and machine learning models.

In [52]:
#load the language model instance in spaCy and disable heavy component for faser processing
nlp = spacy.load("en_core_web_sm", disable = ["parser", "ner", "textcat"])

### Define Stop word

keep Neagtions as they are useful for sentiment analysis

In [53]:
stopwords = nlp.Defaults.stop_words

# keep important negation words for sentiment
negations = {"not", "no", "never", "n't"}

stopwords = stopwords - negations

## Define the cleaning function

This removes:
- HTML tags
- Urlds
- punctuation
- non-alphanumeric characters
- extra whitespace

In [54]:
def clean_review(text):

    text = str(text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)

    # Remove non-alphanumeric characters
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

## Apply basic cleaning

In [55]:
# apply cleaning
data["reviewText_cleaned"] = data["reviewText"].apply(clean_review)

## spaCy Preprocessing function

This performs:
- tokenization
- stopword removal
- lemmatization

In [61]:
def process_doc(doc):

    tokens = []

    for token in doc:
        if token.is_punct or token.is_space or token.like_num:
            continue

        if token.text.lower() in stopwords:
            continue

        tokens.append(token.lemma_.lower())

     # if nothing remains, return None
    if len(tokens) == 0:
        return None

    return " ".join(tokens)    

## Apply spaCy Preprocessing

In [62]:


# spacy batch processing
processed_reviews = []

docs = nlp.pipe(
    data["reviewText_cleaned"],
    batch_size=2000,
    n_process=-1
)

for doc in docs:
    processed_reviews.append(process_doc(doc))

In [63]:
# save the processed 
data["reviewText_processed"] = processed_reviews

# Remove Empty Reviews Before Saving
data = data.dropna(subset=["reviewText_processed"])

# save dataset to CSV
data.to_csv("amazon_reviews_processed.csv", index=False)

In [64]:
# preview the data
data.sample(10)

,asin,category,reviewerID,reviewerName,overall,reviewText,unixReviewTime,reviewTime,verified,vote,has_vote,review_length,word_count,short_review_flag,reviewText_cleaned,reviewText_processed
258182,B000OX89XI,Pet Supplies,A3QE4K9PCML4NE,E. Feurer,4.0,high quality crate with easy setup. I Just wis...,1438214400,"07 30, 2015",True,0,0,72,14,0,high quality crate with easy setup I Just wish...,high quality crate easy setup wish swing open
14644,B00CYQP3AK,Amazon Devices & Accessories,A3RVW73IX5CDSL,Brian K. Watson,5.0,Wife loves it. She has hardly sat it down sinc...,1419724800,"12 28, 2014",True,0,0,142,29,0,Wife loves it She has hardly sat it down since...,wife love hardly sit open christmas day m glad...
547966,0312577222,Books,A1I8T8A3RS3DUR,R. SEEKATZ,5.0,Loved the book. Could not put it down. Chara...,1466726400,"06 24, 2016",True,0,0,77,12,0,Loved the book Could not put it down Character...,love book not characters depict wonderfully
223091,B01BHTSIOC,Movies & TV,ADEOL0TW2KL32,Paul Crabtree,5.0,Very well done in the vain of Downtown Abbey!,1465257600,"06 7, 2016",True,0,0,45,9,0,Very well done in the vain of Downtown Abbey,vain downtown abbey
328900,0007420412,Books,AEMOGGMQTOCSL,C. Cothern,5.0,This book kept me in suspense and wanting more...,1392768000,"02 19, 2014",True,0,0,98,20,0,This book kept me in suspense and wanting more...,book keep suspense want not wait read book series
502379,0141353678,Books,A3O2K869MHTQT3,Karann,5.0,Relationships sometimes can only occur when th...,1402099200,"06 7, 2014",True,0,0,143,21,0,Relationships sometimes can only occur when th...,relationship occur occur never occur circumsta...
531876,0385537859,Books,A3BXWHRWWX5GUZ,Ione Y. DeOllos,5.0,Very nice.,1523750400,"04 15, 2018",True,0,0,10,2,1,Very nice,nice
61970,B00FLYWNYQ,Home & Kitchen,A2TIWH8L1B1MY0,Erica,5.0,My husband got this for me for Christmas and I...,1516838400,"01 25, 2018",True,0,0,127,29,0,My husband got this for me for Christmas and I...,husband get christmas no idea ip omg love use ...
560985,0385537859,Books,A3Q4TYJVAM4IRM,Omar Siddique,4.0,"There's a lot about ""Inferno"" that should rub ...",1427587200,"03 29, 2015",True,0,0,1081,165,0,There s a lot about Inferno that should rub me...,s lot inferno rub wrong way writing simple plo...
246407,B000W5QSYA,Pet Supplies,A2W22UDOQZY9E4,Crystal Terry,5.0,The only dog food I feed my dogs!! They love t...,1502928000,"08 17, 2017",True,0,0,203,39,0,The only dog food I feed my dogs They love thi...,dog food feed dog love stuff good stomach reco...


### Review Text Columns

The dataset contains three versions of the review text to preserve the original data while supporting different stages of preprocessing for analysis.

- **reviewText** contains the original raw review as written by the user. This column is kept unchanged to preserve the original data and allow verification or comparison if needed.

- **reviewText_cleaned** is the review text after the initial data cleaning phase. During this stage, noise such as URLs, HTML tags, punctuation, non-alphanumeric characters, and extra whitespace were removed to standardize the text.

- **reviewText_processed** represents the fully preprocessed text used for Natural Language Processing (NLP) tasks. At this stage, spaCy was used to perform tokenization, lemmatization, and stopword removal to normalize the vocabulary before feature extraction and sentiment modelling.

Maintaining these three versions of the text ensures transparency in the preprocessing pipeline while allowing the original data to remain intact for reference and reproducibility.

## References

The implementation was based on standard Python data preprocessing techniques using pandas and NumPy.

Useful documentation:
- Pandas documentation: data cleaning and missing values
- Pandas string methods documentation
- NumPy documentation for conditional feature creation
-  spaCy was used during the preprocessing stage to perform tokenization, lemmatization, and stopword removal on review text before feature extraction and modelling. These processes help normalize textual data and reduce vocabulary complexity for machine learning models.

Main functions used:
- pd.read_csv()
- dropna()
- .fillna()
- .astype(str)
- .str.strip()
- str.split()

## ChatGPT
ChatGPT was used as a support tool to assist with debugging code, clarifying implementation steps, and providing explanations during the development of the preprocessing and modelling pipeline.

**Reference**

OpenAI. (2025). *ChatGPT (GPT-5.3) [Large language model]*.  Available at: https://chat.openai.com/

Honnibal, M. & Montani, I. (2017). *spaCy 2: Natural language understanding with Bloom embeddings, convolutional neural networks and incremental parsing*. Explosion AI. Available at: https://spacy.io


